In [3]:
from pathlib import Path 
from langchain_core.documents import Document
text = Path("/home/atul/Desktop/langRAG/documents/company_policy_handbook.md").read_text()
docs = [Document(page_content = text , metadata={"source":"company_policy_handbook.md"})]

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter= RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 30
)

chunks = splitter.split_documents(docs)

len(chunks)

25

In [5]:
for i, chunk in enumerate(chunks,start = 1):
    print("="*80)
    print("Chunk:",i)
    print(chunk.page_content)

Chunk: 1
# Company Policy Handbook

## Leave Policy
Chunk: 2
Full-time employees receive 24 paid leave days per calendar year. Unused leave can be carried forward into the next calendar year, but the maximum carry-forward balance is 10 days. Any unused leave
Chunk: 3
is 10 days. Any unused leave above 10 days expires at the end of the year.
Chunk: 4
Part-time employees receive paid leave on a prorated basis according to their weekly working hours. Part-time employees can carry forward up to 5 unused leave days into the next calendar year.
Chunk: 5
Contractors are not eligible for paid leave and cannot carry forward unused leave days. Contractors may request unpaid time away from work, but the request must be approved by their project manager.
Chunk: 6
Employees should submit planned leave requests at least 10 working days before the start date. Emergency leave can be reported as soon as reasonably possible through the HR portal or by notifying the
Chunk: 7
HR portal or by notifying the

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings


embedding_model = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-small-en-v1.5",
    encode_kwargs = {"normalize_embeddings": True}
)

/home/atul/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5050.41it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
sample_text = chunks[0].page_content

vector = embedding_model.embed_query(sample_text)

print(type(vector))
print(len(vector))
print(vector[0:10])

<class 'list'>
384
[-0.014960082247853279, -0.008256953209638596, 0.03871187940239906, -0.047703370451927185, 0.0608067512512207, 0.060997866094112396, 0.01637531816959381, -0.01280826237052679, 0.004615106154233217, -0.012843075208365917]


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

text_1 = chunks[0].page_content
text_2 = chunks[1].page_content

vec_1 = embedding_model.embed_query(text_1)
vec_2 = embedding_model.embed_query(text_2)

score = cosine_similarity([vec_1],[vec_2])[0][0]

print(score)

0.6628016381461657


In [9]:
query = "Can contractors carry forword unused leave?"
query_vec = embedding_model.embed_query(query)

for i, chunk in enumerate(chunks,start=1):
    chunk_vec = embedding_model.embed_query(chunk.page_content[:120])
    score = cosine_similarity([query_vec],[chunk_vec])[0][0]
    print(i, round(score,4),chunk.page_content[:120])

1 0.7166 # Company Policy Handbook

## Leave Policy
2 0.7283 Full-time employees receive 24 paid leave days per calendar year. Unused leave can be carried forward into the next cale
3 0.7521 is 10 days. Any unused leave above 10 days expires at the end of the year.
4 0.6636 Part-time employees receive paid leave on a prorated basis according to their weekly working hours. Part-time employees 
5 0.8227 Contractors are not eligible for paid leave and cannot carry forward unused leave days. Contractors may request unpaid t
6 0.6862 Employees should submit planned leave requests at least 10 working days before the start date. Emergency leave can be re
7 0.5866 HR portal or by notifying the employee's manager.
8 0.6438 ## Remote Work Policy
9 0.6341 Full-time and part-time employees may work remotely up to 3 days per week with manager approval. The employee must remai
10 0.4709 10:00 AM to 4:00 PM in their local time zone.
11 0.7223 Contractors may work remotely only if remote work is inclu

In [10]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding = embedding_model,
    collection_name = "company_policy"
)


In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2} )

In [12]:
query = "Can contractors carry forward unused leave?"

results = retriever.invoke(query)

for i,docs in enumerate (results,start = 1):
    print("="*80)
    print("Rank:",i)
    print(docs.page_content)
    print(docs.metadata)

Rank: 1
Contractors are not eligible for paid leave and cannot carry forward unused leave days. Contractors may request unpaid time away from work, but the request must be approved by their project manager.
{'source': 'company_policy_handbook.md'}
Rank: 2
is 10 days. Any unused leave above 10 days expires at the end of the year.
{'source': 'company_policy_handbook.md'}


In [16]:
query = "Can contractors carry forward unused leave?"

retrieved_docs = retriever.invoke(query)

for i in retrieved_docs:
    print(docs.page_content)

is 10 days. Any unused leave above 10 days expires at the end of the year.
is 10 days. Any unused leave above 10 days expires at the end of the year.


In [18]:
context = "\n\n".join(docs.page_content for docs in retrieved_docs)
print(context)

Contractors are not eligible for paid leave and cannot carry forward unused leave days. Contractors may request unpaid time away from work, but the request must be approved by their project manager.

is 10 days. Any unused leave above 10 days expires at the end of the year.


In [19]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:1.5b"
)

In [20]:
prompt = f"""
You are a helpful assistant.

Answer the question only using the provided context.
If the answer is not in the context, say:
"I don not know from the provided context."

Context:
{context}

Question:
{query}

Answer:
"""

In [22]:
response = llm.invoke(prompt)
print(response.content)

No, contractors cannot carry forward unused leave beyond 10 days as any unused leave over that period expires at the end of the year.


In [32]:
def ask_rag(query,docs,llm):
    retriever_docs = retriever.invoke(query)
    context= "\n\n".join(docs.page_content for docs in retriever_docs)
    prompt = f"""
    You are a helpful assistant.

    Answer the question only using the provided context.
    If the answer is not in the context, say:
    "I don not know from the provided context."

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    response = llm.invoke(prompt)
    print(response.content)
    

In [ ]:
query_1 = "When must a lost laptop be reported to IT?"

ask_rag(query,docs,llm)

Lost or stolen devices must be reported to IT within 2 hours of discovery.
